# Finance AI Analysis

This notebook builds a first-pass finance occupation ranking for the question:

**Which business and financial occupations are most likely to be enhanced by AI?**

Notes:
- We use O*NET occupation features and the AIOE occupation exposure data.
- The direct BLS bulk downloads returned `403 Access Denied`, so this notebook uses manually entered BLS Occupational Outlook Handbook metrics for the shortlist where available.
- The final `ai_enhancement_score` is a heuristic ranking, not a causal estimate.


In [1]:
%pip install pandas openpyxl


Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import zipfile
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/stephenye/stephen/coding/project/datajam')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports'

ONET_ZIP = RAW_DIR / 'onet_30_2_excel.zip'
AIOE_XLSX = RAW_DIR / 'aioe_data.xlsx'

FOCUS_OCCUPATIONS = {
    '13-1031': {'focus_occupation_title': 'Claims Adjusters, Examiners, and Investigators', 'group_label': 'routine'},
    '13-2011': {'focus_occupation_title': 'Accountants and Auditors', 'group_label': 'analytical'},
    '13-2031': {'focus_occupation_title': 'Budget Analysts', 'group_label': 'analytical'},
    '13-2041': {'focus_occupation_title': 'Credit Analysts', 'group_label': 'analytical'},
    '13-2051': {'focus_occupation_title': 'Financial and Investment Analysts', 'group_label': 'analytical'},
    '13-2052': {'focus_occupation_title': 'Personal Financial Advisors', 'group_label': 'analytical'},
    '13-2053': {'focus_occupation_title': 'Insurance Underwriters', 'group_label': 'routine'},
    '13-2054': {'focus_occupation_title': 'Financial Risk Specialists', 'group_label': 'analytical'},
    '13-2072': {'focus_occupation_title': 'Loan Officers', 'group_label': 'routine'},
}

BLS_MANUAL_METRICS = {
    '13-1031': {
        'median_annual_wage_2024': 76790,
        'projected_growth_pct_2024_2034': -5,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/claims-adjusters-appraisers-examiners-and-investigators.htm',
    },
    '13-2011': {
        'median_annual_wage_2024': 81680,
        'projected_growth_pct_2024_2034': 5,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/accountants-and-auditors.htm',
    },
    '13-2031': {
        'median_annual_wage_2024': 87930,
        'projected_growth_pct_2024_2034': 1,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/budget-analysts.htm',
    },
    '13-2041': {
        'median_annual_wage_2024': 80970,
        'projected_growth_pct_2024_2034': -4,
        'source_url': 'https://www.onetonline.org/link/summary/13-2041.00',
    },
    '13-2051': {
        'median_annual_wage_2024': 101350,
        'projected_growth_pct_2024_2034': 6,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/financial-analysts.htm',
    },
    '13-2052': {
        'median_annual_wage_2024': 102140,
        'projected_growth_pct_2024_2034': 10,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/personal-financial-advisors.htm',
    },
    '13-2053': {
        'median_annual_wage_2024': 79880,
        'projected_growth_pct_2024_2034': -3,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/insurance-underwriters.htm',
    },
    '13-2054': {
        'median_annual_wage_2024': 106000,
        'projected_growth_pct_2024_2034': 7,
        'source_url': 'https://www.onetonline.org/link/localtrends/13-2054.00',
    },
    '13-2072': {
        'median_annual_wage_2024': 74180,
        'projected_growth_pct_2024_2034': 2,
        'source_url': 'https://www.bls.gov/ooh/business-and-financial/loan-officers.htm',
    },
}

COGNITIVE_SKILLS = [
    'Reading Comprehension',
    'Active Listening',
    'Critical Thinking',
    'Complex Problem Solving',
    'Judgment and Decision Making',
    'Mathematics',
]

FINANCE_KNOWLEDGE = [
    'Economics and Accounting',
    'Mathematics',
]

ROUTINE_KNOWLEDGE = [
    'Administrative',
    'Administration and Management',
]

ANALYTICAL_WORK_ACTIVITIES = [
    'Analyzing Data or Information',
]

ROUTINE_WORK_ACTIVITIES = [
    'Processing Information',
    'Documenting/Recording Information',
]

WEIGHTS = {
    'ai_exposure_z': 0.25,
    'job_zone_z': 0.10,
    'cognitive_skill_level_z': 0.15,
    'finance_knowledge_level_z': 0.10,
    'analytical_activity_level_z': 0.10,
    'median_wage_z': 0.15,
    'growth_z': 0.10,
    'routine_knowledge_level_z': -0.025,
    'routine_activity_level_z': -0.025,
}


In [3]:
def normalize_soc(series):
    return series.astype(str).str.extract(r'(\d{2}-\d{4})', expand=False)


def read_onet_excel_from_zip(zip_file, file_name, **kwargs):
    with zip_file.open(f'db_30_2_excel/{file_name}') as handle:
        return pd.read_excel(io.BytesIO(handle.read()), **kwargs)


def aggregate_scale_average(table, element_names, scale_id, column_name):
    filtered = table[
        table['Element Name'].isin(element_names)
        & (table['Scale ID'] == scale_id)
    ].copy()
    if filtered.empty:
        return pd.Series(dtype=float, name=column_name)
    return filtered.groupby('soc_code')['Data Value'].mean().rename(column_name)


def zscore(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std


In [4]:
# Load AIOE occupation exposure data
aioe = pd.read_excel(AIOE_XLSX, sheet_name='Appendix A')
aioe['soc_code'] = normalize_soc(aioe['SOC Code'])
aioe = aioe.rename(
    columns={
        'Occupation Title': 'aioe_occupation_title',
        'AIOE': 'ai_exposure',
    }
)[['soc_code', 'aioe_occupation_title', 'ai_exposure']]

# Load the O*NET tables we need from the zip archive
with zipfile.ZipFile(ONET_ZIP) as onet_zip:
    occupation_data = read_onet_excel_from_zip(onet_zip, 'Occupation Data.xlsx')
    job_zones = read_onet_excel_from_zip(onet_zip, 'Job Zones.xlsx')
    skills = read_onet_excel_from_zip(onet_zip, 'Skills.xlsx')
    knowledge = read_onet_excel_from_zip(onet_zip, 'Knowledge.xlsx')
    work_activities = read_onet_excel_from_zip(onet_zip, 'Work Activities.xlsx')

for df, col in [
    (occupation_data, 'O*NET-SOC Code'),
    (job_zones, 'O*NET-SOC Code'),
    (skills, 'O*NET-SOC Code'),
    (knowledge, 'O*NET-SOC Code'),
    (work_activities, 'O*NET-SOC Code'),
]:
    df['soc_code'] = normalize_soc(df[col])

print('AIOE rows:', len(aioe))
print('O*NET occupation rows:', len(occupation_data))


AIOE rows: 774
O*NET occupation rows: 1016


In [5]:
# Keep the Business and Financial Operations family (SOC 13-xxxx)
family = occupation_data[
    occupation_data['soc_code'].str.startswith('13-', na=False)
].copy()

family = (
    family.sort_values(['soc_code', 'O*NET-SOC Code'])
    .drop_duplicates('soc_code')
    .rename(columns={'Title': 'occupation_title'})[['soc_code', 'occupation_title']]
    .reset_index(drop=True)
)

job_zone_summary = job_zones.groupby('soc_code')['Job Zone'].mean().rename('job_zone')

composites = [
    aggregate_scale_average(skills, COGNITIVE_SKILLS, 'LV', 'cognitive_skill_level'),
    aggregate_scale_average(knowledge, FINANCE_KNOWLEDGE, 'LV', 'finance_knowledge_level'),
    aggregate_scale_average(knowledge, ROUTINE_KNOWLEDGE, 'LV', 'routine_knowledge_level'),
    aggregate_scale_average(work_activities, ANALYTICAL_WORK_ACTIVITIES, 'LV', 'analytical_activity_level'),
    aggregate_scale_average(work_activities, ROUTINE_WORK_ACTIVITIES, 'LV', 'routine_activity_level'),
]

manual = pd.DataFrame([
    {'soc_code': soc_code, **metrics}
    for soc_code, metrics in BLS_MANUAL_METRICS.items()
])

focus_meta = pd.DataFrame([
    {'soc_code': soc_code, **metadata}
    for soc_code, metadata in FOCUS_OCCUPATIONS.items()
])

merged = family.merge(aioe, on='soc_code', how='left')
merged = merged.merge(job_zone_summary, on='soc_code', how='left')

for series in composites:
    merged = merged.merge(series, on='soc_code', how='left')

merged = merged.merge(manual, on='soc_code', how='left')
merged = merged.merge(focus_meta, on='soc_code', how='left')
merged['occupation_title'] = (
    merged['focus_occupation_title']
    .fillna(merged['occupation_title'])
    .fillna(merged['aioe_occupation_title'])
)

merged.head()


,soc_code,occupation_title,aioe_occupation_title,ai_exposure,job_zone,cognitive_skill_level,finance_knowledge_level,routine_knowledge_level,analytical_activity_level,routine_activity_level,median_annual_wage_2024,projected_growth_pct_2024_2034,source_url,focus_occupation_title,group_label


In [6]:
# Standardize the numeric inputs
merged['median_wage_z'] = zscore(merged['median_annual_wage_2024'])
merged['growth_z'] = zscore(merged['projected_growth_pct_2024_2034'])

for col in [
    'ai_exposure',
    'job_zone',
    'cognitive_skill_level',
    'finance_knowledge_level',
    'routine_knowledge_level',
    'analytical_activity_level',
    'routine_activity_level',
]:
    merged[f'{col}_z'] = zscore(merged[col])

# Weighted score using only the values that exist for each row.
# This keeps financial analysts in the table even though some O*NET ratings are missing.
weighted_sum = pd.Series(0.0, index=merged.index)
weight_total = pd.Series(0.0, index=merged.index)

for col, weight in WEIGHTS.items():
    values = merged[col]
    present = values.notna()
    weighted_sum.loc[present] += values.loc[present] * weight
    weight_total.loc[present] += abs(weight)

merged['ai_enhancement_score'] = weighted_sum / weight_total.replace(0, pd.NA)

# Require the AI exposure metric for a final score.
merged.loc[merged['ai_exposure'].isna(), 'ai_enhancement_score'] = pd.NA

merged['ai_enhancement_rank'] = (
    merged['ai_enhancement_score']
    .rank(ascending=False, method='min')
    .astype('Int64')
)


In [7]:
# Focus on the finance shortlist
focus = merged[merged['soc_code'].isin(FOCUS_OCCUPATIONS.keys())].copy()
focus = focus.sort_values('ai_enhancement_score', ascending=False, na_position='last').reset_index(drop=True)
focus['focus_rank'] = range(1, len(focus) + 1)

display_cols = [
    'focus_rank',
    'soc_code',
    'occupation_title',
    'group_label',
    'ai_exposure',
    'job_zone',
    'cognitive_skill_level',
    'finance_knowledge_level',
    'routine_activity_level',
    'median_annual_wage_2024',
    'projected_growth_pct_2024_2034',
    'ai_enhancement_score',
]

focus[display_cols]


,focus_rank,soc_code,occupation_title,group_label,ai_exposure,job_zone,cognitive_skill_level,finance_knowledge_level,routine_activity_level,median_annual_wage_2024,projected_growth_pct_2024_2034,ai_enhancement_score


In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

family_out = PROCESSED_DIR / 'business_financial_family_features.csv'
focus_out = PROCESSED_DIR / 'finance_focus_jobs.csv'
report_out = REPORTS_DIR / 'finance_job_ranking.md'

merged.sort_values('ai_enhancement_score', ascending=False).to_csv(family_out, index=False)
focus.to_csv(focus_out, index=False)

lines = [
    '# Finance AI Job Ranking',
    '',
    'This first-pass ranking uses O*NET occupation features plus the Felten-Raj-Seamans AIOE score.',
    'BLS bulk download files were blocked, so wage and growth values were manually copied from official BLS Occupational Outlook Handbook pages for the shortlist where available.',
    '',
    '## Ranked Finance Occupations',
    '',
    '| Rank | SOC | Occupation | Group | AI Exposure | Enhancement Score | Median Wage 2024 | Growth 2024-34 |',
    '| --- | --- | --- | --- | ---: | ---: | ---: | ---: |',
]

for _, row in focus.iterrows():
    wage = '' if pd.isna(row['median_annual_wage_2024']) else f"{int(row['median_annual_wage_2024']):,}"
    growth = '' if pd.isna(row['projected_growth_pct_2024_2034']) else f"{row['projected_growth_pct_2024_2034']:.0f}%"
    exposure = '' if pd.isna(row['ai_exposure']) else f"{row['ai_exposure']:.3f}"
    score = '' if pd.isna(row['ai_enhancement_score']) else f"{row['ai_enhancement_score']:.3f}"
    lines.append(
        f"| {int(row['focus_rank'])} | {row['soc_code']} | {row['occupation_title']} | {row['group_label']} | {exposure} | {score} | {wage} | {growth} |"
    )

lines.extend([
    '',
    '## Notes',
    '',
    '- The `ai_enhancement_score` is a heuristic ranking, not a causal estimate.',
    '- Higher scores reflect stronger AI exposure combined with more analytical and higher-complexity signals.',
    '- Occupations missing the AIOE exposure value are left unranked in this first pass.',
    '',
    '## Official BLS Source Pages Used For Manual Metrics',
    '',
])

seen_urls = []
for url in focus['source_url'].dropna():
    if url not in seen_urls:
        seen_urls.append(url)
        lines.append(f'- {url}')

report_out.write_text('\n'.join(lines) + '\n', encoding='utf-8')

print('Saved:')
print(family_out)
print(focus_out)
print(report_out)


Saved:
/Users/stephenye/stephen/coding/project/datajam/data/processed/business_financial_family_features.csv
/Users/stephenye/stephen/coding/project/datajam/data/processed/finance_focus_jobs.csv
/Users/stephenye/stephen/coding/project/datajam/reports/finance_job_ranking.md
